## Connect with pandas and sqlalchemy

In [4]:
import os
import pandas as pd
from sqlalchemy import create_engine

engine = create_engine(
    f"postgresql+psycopg2://{os.getenv('POSTGRES_USER')}:"
    f"{os.getenv('POSTGRES_PASSWORD')}@db:5432/"
    f"{os.getenv('POSTGRES_DB')}"
)

In [5]:
df = pd.read_sql(
    "SELECT * FROM strava_bronze.activities LIMIT 10;",
    con=engine
)

In [6]:
df.head()

,id,name,start_date,distance,duration,activity_type,raw_data
0,18252667108,Afternoon Mountain Bike Ride,2026-04-25 12:42:45,19440.0,0 days 01:17:31,Ride,"{'id': 18252667108, 'map': {'id': 'a1825266710..."
1,18214252549,Afternoon Mountain Bike Ride,2026-04-22 14:54:11,17714.0,0 days 01:48:27,Ride,"{'id': 18214252549, 'map': {'id': 'a1821425254..."
2,18172706469,Afternoon Mountain Bike Ride,2026-04-19 12:26:36,19452.0,0 days 01:21:42,Ride,"{'id': 18172706469, 'map': {'id': 'a1817270646..."
3,18147057670,Afternoon Mountain Bike Ride,2026-04-17 14:46:48,19513.0,0 days 01:39:15,Ride,"{'id': 18147057670, 'map': {'id': 'a1814705767..."
4,18025363161,Lunch Mountain Bike Ride,2026-04-08 10:14:20,12096.0,0 days 01:02:05,Ride,"{'id': 18025363161, 'map': {'id': 'a1802536316..."


## Connect with pyspark

In [2]:
from pyspark.sql import SparkSession
import os

spark = SparkSession.builder \
    .appName("PostgresTest") \
    .config("spark.jars.packages", "org.postgresql:postgresql:42.7.3") \
    .getOrCreate()

df = spark.read \
    .format("jdbc") \
    .option("url", f"jdbc:postgresql://db:5432/{os.getenv('POSTGRES_DB')}") \
    .option("dbtable", "strava_bronze.activities") \
    .option("user", os.getenv("POSTGRES_USER")) \
    .option("password", os.getenv("POSTGRES_PASSWORD")) \
    .option("driver", "org.postgresql.Driver") \
    .load()

df.show()

+-----------+--------------------+-------------------+--------+--------+-------------+--------------------+
|         id|                name|         start_date|distance|duration|activity_type|            raw_data|
+-----------+--------------------+-------------------+--------+--------+-------------+--------------------+
|18252667108|Afternoon Mountai...|2026-04-25 12:42:45| 19440.0|01:17:31|         Ride|{"id": 1825266710...|
|18214252549|Afternoon Mountai...|2026-04-22 14:54:11| 17714.0|01:48:27|         Ride|{"id": 1821425254...|
|18172706469|Afternoon Mountai...|2026-04-19 12:26:36| 19452.0|01:21:42|         Ride|{"id": 1817270646...|
|18147057670|Afternoon Mountai...|2026-04-17 14:46:48| 19513.0|01:39:15|         Ride|{"id": 1814705767...|
|18025363161|Lunch Mountain Bi...|2026-04-08 10:14:20| 12096.0|01:02:05|         Ride|{"id": 1802536316...|
|18015231296|Afternoon Mountai...|2026-04-07 14:42:50| 19557.0|01:43:01|         Ride|{"id": 1801523129...|
|17939355572|Afternoon Mount

## Use sql magic

In [2]:
import os

conn_str = (
    f"postgresql://{os.getenv('POSTGRES_USER')}:"
    f"{os.getenv('POSTGRES_PASSWORD')}@"
    f"{os.getenv('POSTGRES_HOST', 'db')}:"
    f"{os.getenv('POSTGRES_PORT', '5432')}/"
    f"{os.getenv('POSTGRES_DB')}"
)

%load_ext sql
%sql $conn_str

In [2]:
%%sql
SELECT * FROM strava_bronze.activities LIMIT 10;

 * postgresql://strava:***@db:5432/strava_db
10 rows affected.


id,name,start_date,distance,duration,activity_type,raw_data
18252667108,Afternoon Mountain Bike Ride,2026-04-25 12:42:45,19440.0,1:17:31,Ride,"{'id': 18252667108, 'map': {'id': 'a18252667108', 'resource_state': 2, 'summary_polyline': '_``aHiz{rBIxAB^MtFgAvIAv@Nb@`Bj@WpDLXSjBJdAA`Ac@jGeB`SqAjQCxBGx@K`@UfDi@xDcA~EoAfE?b@G`@Yn@QrA_@dB[`CM`@e@|DC`BGb@yCbIm@bDCf@o@zBWfACp@[t@q@xCsBlHWjAQ|BEzB^HGhCG\\mBrDOj@kAlBu@`BdCcBz@WlAq@bD_AZWTo@\\mBbAuCz@kDvA_C|@_Cj@qBTa@FHe@zBUbD_AfDKz@AdBe@jCE`BNnF]|@STuAVc@XcFpFyAlC_@bA}@tAc@`AMb@@\\Td@p@NpCMzBi@t@g@`AoAxB{@pAoB|AeArCyCTaAC_BFiBZcFR{EVgCf@qCp@{B`@yBXy@Ze@xAs@rC}BxA_A`@SPJwA|@aDfDcB`Aa@d@Qb@w@~Du@hCS~CQnHJb@NZt@n@n@A`@M`CoBrAqBtCmDjAoB`C}EdBsAjBm@{A\\KhAIT[\\uAbAwBlFcBzCsD~D}AnAgFfG_A`AM^CVHJ|AQhADTLRh@A^Yp@ChBDxCXtB?n@MfBW~@SrB[f@gAn@qAhAuBw@w@MUBi@pAa@r@u@~Bc@z@k@|BWj@cArA}@pCKd@KrADvBv@pBfArBhCbHa@~@mB`@aAh@oANWRGPGh@ArEM~AYrAeB~COb@YxBMdBm@tCKdBWLoBBgBc@gAZoAE}@Zo@IuAXuC^}CDaANe@Gc@N{BvBy@TeA[_BaA_AY_CJcHjAoBEuA[{Aq@gGsAkDkAqBc@cOa@gJ}@KIQw@_@Qu@w@w@cAc@}@_AcAiAcBs@g@s@_AIq@?cBUkA?k@h@k@zAs@ZY|BeEtEgG~BeC~AoAf@Dz@d@|Ad@~AWt@}@\\{@|@qHFEn@d@p@AlBaAXa@Ru@@k@S@w@pAiAx@o@EQW?_@p@mFHeB]wFVqGOwDk@eBUyAE{BJ{AB}AMkD@i@HYj@]DOFaBt@yF\\qA~@iEl@gBVg@`@kBL{Bf@sCb@mATmAf@eAb@yAJi@?y@NaBxBoEZsAfBuD`AaDfBsDOs@eEuJM_@@]OCMZOQoFaNgA{CEWIO[p@MACO\\_AtA{BzEiFbDmEfKoL~@w@fHqI`@]b@KbG{EvJgHrAw@|HaInKmJ~IwIdFmItGuJdPkXJRAh@]fBUfC@hBHvA'}, 'name': 'Afternoon Mountain Bike Ride', 'type': 'Ride', 'manual': False, 'athlete': {'id': 42803987, 'resource_state': 1}, 'commute': False, 'flagged': False, 'gear_id': None, 'private': False, 'trainer': False, 'distance': 19440.0, 'elev_low': 153.2, 'pr_count': 6, 'timezone': '(GMT+01:00) Europe/Budapest', 'elev_high': 492.8, 'max_speed': 11.6, 'upload_id': 19355714532, 'end_latlng': [47.52, 18.99], 'has_kudoed': False, 'kilojoules': 455.2, 'sport_type': 'MountainBikeRide', 'start_date': '2026-04-25T12:42:45Z', 'utc_offset': 7200.0, 'visibility': 'everyone', 'device_name': 'Suunto 9 Peak Pro', 'external_id': 'stripped_69ecc9202b8f24188762dc01.fit', 'kudos_count': 0, 'moving_time': 4322, 'photo_count': 0, 'average_temp': 23, 'device_watts': False, 'elapsed_time': 4651, 'start_latlng': [47.52, 18.99], 'workout_type': None, 'athlete_count': 1, 'average_speed': 4.498, 'average_watts': 105.3, 'comment_count': 0, 'has_heartrate': False, 'location_city': None, 'upload_id_str': '19355714532', 'location_state': None, 'resource_state': 2, 'location_country': None, 'start_date_local': '2026-04-25T14:42:45Z', 'achievement_count': 16, 'from_accepted_tag': False, 'heartrate_opt_out': False, 'total_photo_count': 0, 'total_elevation_gain': 428.0, 'display_hide_heartrate_option': False}"
18214252549,Afternoon Mountain Bike Ride,2026-04-22 14:54:11,17714.0,1:48:27,Ride,"{'id': 18214252549, 'map': {'id': 'a18214252549', 'resource_state': 2, 'summary_polyline': 'gx_aHwx|rBHsEv@}DEUOKc@D{@y@iLsPmAyBc@o@MAi@}@qBeBk@aAyB}BUEcCwB]v@k@hCoCbKiA~Du@bB{@tAsBbB}Ct@}B[cBaAe@g@q@sAa@yB[m@m@a@a@Ek@Jg@j@uAxFsDlRC~VGzDqAxKQt@ORc@K{CyCs@cAYcABiBI}Bs@gCKaA]uAs@mBwByD[]qBpAq@UqH^mDQcDjFmAbByGvBgBjAOb@Cb@[r@u@`Au@XaAJc@j@[DoA|@[x@ONi@CmBXk@Vu@?kClBu@CWRcA|AgAfAiAr@gBx@c@LgAs@w@_Ac@KwI|@iKNwCGqA_@qCyA_Co@iCUoBJgDg@oDmBO[OcA[_F?q@ZuA`@cAnAeBVy@xAkBpAL|@K@Jg@tAsA~@p@q@RERYf@iANe@Ca@FMQzAq@tAyAnAz@aAb@YVu@b@o@@{@Gz@u@fBcAj@q@z@TWi@Kl@?p@o@f@SRq@h@_AAm@DIKrAs@tAsBnBp@aAn@WVg@Jc@f@{@DoAFWb@aAAi@F]ZkAyAvGEKx@yBD_ATk@Fg@qAbG@f@Ck@Ns@b@s@Au@Z}@Bc@[~AIvAc@dB@Vr@DLHEDHWXFT[^JDQLAw@T?IFEVJLQJ?qA^JDV]PLNUJ@oAl@{AQKS?SAx@m@tAgChC]lAeAd@Wf@g@Cg@fAYFC\\Dr@Mk@UVa@QM_@Y?Ki@KKLJTr@FKJHYESk@Vr@NKb@t@VALQLl@R~Db@|AjDrFpA|CnCvHPVbAK`@c@~A[xCAxHkAl@A^Z~BEdAa@r@KnEeBpDKGv@_@pAyAbC_A~@@Pz@vCNvAWjD_AjH}@jB{FzIqA|Bw@`C?Vj@b@OxATfBNXdBj@X`@TfBtArAp@bCdChDZdBlAdE|CzAz@lAtAvCDjAj@bD|@vAv@x@|BdA|BzB\\NvA[t@FtASnA@n@OlCBj@M\\]r@mALc@t@_GB{An@}GZkBx@kCBo@MoCJw@l@}Bz@iBr@mDdA}A`EsBf@m@vAcErAiGrAiCjBwA`AqAZaAb@uB`@kC`@}@r@cAtCuC`BgAT@^j@~@?|VcBn@Ml@g@Z\\V@fFoF|QkPfBsBnBcBzAaCr@aB`@a@xHkLtPcYJX?Zk@fDCpDFnA'}, 'name': 'Afternoon Mountain Bike Ri

In [45]:
%%sql
SELECT COUNT(*) FROM strava_bronze.activities_detailed;

 * postgresql://strava:***@db:5432/strava_db
1 rows affected.


count
693


In [3]:
%%sql
SELECT MIN(start_date) FROM public_strava_silver.segment_effort;

 * postgresql://strava:***@db:5432/strava_db
1 rows affected.


min
2026-01-25 15:06:12+00:00
